# ⚡ Enterprise Text-to-SQL: 4-Bit QLoRA Fine-Tuning on NVIDIA GPUs
### Domain Adaptation of `Qwen/Qwen2.5-Coder-7B-Instruct` with Hugging Face, PEFT, TRL & bitsandbytes NF4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asmohamedarfeen/fine-tuning-qlora/blob/main/notebooks/colab_qlora_sql.ipynb)

This notebook trains a high-precision Text-to-SQL engine on an NVIDIA GPU (Google Colab T4 / A100 / L4) using:
- **Base Model:** `Qwen/Qwen2.5-Coder-7B-Instruct`
- **Quantization:** 4-bit NormalFloat (`bitsandbytes` NF4 with double quantization)
- **LoRA Hyperparameters:** Rank `r=16`, `alpha=32`, `dropout=0.05` across all linear projection layers
- **Trainer:** Hugging Face `trl.SFTTrainer` with ChatML formatted enterprise multi-table schemas
- **Evaluation:** SQLite in-memory sandbox and `sqlglot` execution benchmark

## 1. Setup & Environment Verification
Check available NVIDIA GPU hardware and install necessary training dependencies.

In [ ]:
# Verify GPU hardware
!nvidia-smi

# Install dependencies for Hugging Face QLoRA & evaluation
!pip install -q -U torch transformers peft trl bitsandbytes datasets accelerate sqlglot

## 2. Configuration & Hyperparameters
Define QLoRA fine-tuning parameters matching `configs/hf_train_config.yaml`.

In [ ]:
import torch

CONFIG = {
    "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
    "output_dir": "adapters/hf_sql_lora",
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "learning_rate": 2.0e-4,
    "batch_size": 2,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 3,
    "warmup_ratio": 0.05,
    "logging_steps": 10,
    "save_steps": 200,
}

print("[*] Training Configuration:")
for key, val in CONFIG.items():
    print(f"    {key}: {val}")

## 3. 4-Bit NF4 Quantization & Model Loading
Configure `bitsandbytes` 4-bit NormalFloat quantization with double quantization.

In [ ]:
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit NF4 Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading tokenizer for {CONFIG['model_id']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[*] Loading 4-bit base model for {CONFIG['model_id']}...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
print("[+] Base model loaded and prepared for k-bit LoRA training.")

## 4. LoRA Adapter Configuration
Set up Low-Rank Adaptation (LoRA) targeting all attention and MLP projection layers.

In [ ]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=CONFIG["target_modules"],
)

print("[*] PEFT LoRA Configuration Initialized:")
print(
    f"    Rank (r): {peft_config.r}, Alpha: {peft_config.lora_alpha}, Dropout: {peft_config.lora_dropout}"
)
print(f"    Target modules: {peft_config.target_modules}")

## 5. Dataset Loading & ChatML Formatting
Load processed JSONL datasets containing multi-turn ChatML messages.

In [ ]:
import os

from datasets import Dataset, DatasetDict, load_dataset

dataset_dir = "data/processed"

if os.path.exists(os.path.join(dataset_dir, "train.jsonl")):
    print("[*] Loading datasets from data/processed...")
    dataset = load_dataset(
        "json",
        data_files={
            "train": os.path.join(dataset_dir, "train.jsonl"),
            "valid": os.path.join(dataset_dir, "valid.jsonl"),
            "test": os.path.join(dataset_dir, "test.jsonl"),
        },
    )
else:
    print(
        "[*] Generating in-memory enterprise Text-to-SQL ChatML dataset for Colab demo..."
    )
    sample_records = [
        {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an expert enterprise SQL database engineer. Given the database schema (DDL), constraints, and natural language question, generate the most efficient, syntactically correct SQL query. Always explain your reasoning briefly before outputting the final SQL block enclosed in ```sql ... ```.",
                },
                {
                    "role": "user",
                    "content": "### Database Schema:\nCREATE TABLE orders (order_id INT PRIMARY KEY, customer_id INT, status VARCHAR(20), total_amount NUMERIC(10,2), created_at TIMESTAMP);\nCREATE TABLE order_items (item_id INT PRIMARY KEY, order_id INT, product_id INT, quantity INT, unit_price NUMERIC(10,2));\n\n### Question:\nFind total revenue and completed order count for customer 1042.",
                },
                {
                    "role": "assistant",
                    "content": "Filter orders by customer_id = 1042 and status = 'completed', then aggregate SUM(total_amount) and COUNT(order_id).\n\n```sql\nSELECT COUNT(order_id) AS completed_orders, COALESCE(SUM(total_amount), 0) AS total_revenue FROM orders WHERE customer_id = 1042 AND status = 'completed';\n```",
                },
            ]
        },
        {
            "messages": [
                {
                    "role": "system",
                    "content": "You are an expert enterprise SQL database engineer. Given the database schema (DDL), constraints, and natural language question, generate the most efficient, syntactically correct SQL query. Always explain your reasoning briefly before outputting the final SQL block enclosed in ```sql ... ```.",
                },
                {
                    "role": "user",
                    "content": "### Database Schema:\nCREATE TABLE subscriptions (sub_id INT PRIMARY KEY, org_id INT, plan_tier VARCHAR(50), mrr NUMERIC(10,2), churned BOOLEAN);\n\n### Question:\nCalculate total active MRR grouped by plan tier.",
                },
                {
                    "role": "assistant",
                    "content": "Filter for active subscriptions (churned = 0) and group by plan_tier, summing MRR.\n\n```sql\nSELECT plan_tier, SUM(mrr) AS active_mrr FROM subscriptions WHERE churned = 0 GROUP BY plan_tier ORDER BY active_mrr DESC;\n```",
                },
            ]
        },
    ]
    dataset = DatasetDict(
        {
            "train": Dataset.from_list(sample_records),
            "valid": Dataset.from_list(sample_records),
            "test": Dataset.from_list(sample_records),
        }
    )

print(f"[+] Dataset ready: {dataset}")

## 6. SFTTrainer Execution Loop
Initialize Hugging Face `SFTTrainer` and execute 4-bit QLoRA fine-tuning.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_train_epochs=CONFIG["num_train_epochs"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    max_length=CONFIG["max_seq_length"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_dir=f"{CONFIG['output_dir']}/logs",
    report_to="none",
    save_total_limit=2,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset.get("valid"),
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

print("[*] Starting SFTTrainer training loop...")
trainer.train()

print(f"[*] Saving fine-tuned adapter weights to {CONFIG['output_dir']}...")
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("[+] LoRA adapter weights saved successfully.")

## 7. Execution Accuracy & SQL Benchmark Sandbox
Verify generated queries against in-memory SQLite sandbox with `sqlglot` syntax validation.

In [ ]:
import sqlite3

import sqlglot

print("[*] Running SQL Validation & Execution Sandbox Benchmark...")


def validate_sql(query: str, dialect: str = "sqlite") -> bool:
    """Validates SQL query syntax using sqlglot."""
    try:
        sqlglot.transpile(query, read=dialect, write=dialect)
        return True
    except (sqlglot.errors.ParseError, sqlglot.errors.TokenError) as exc:
        print(f"[!] Syntax error: {exc}")
        return False


def execute_in_sandbox(ddl: str, seed_sql: str, test_query: str):
    """Executes query in isolated in-memory SQLite sandbox."""
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    try:
        cursor.executescript(ddl)
        if seed_sql:
            cursor.executescript(seed_sql)
        cursor.execute(test_query)
        results = cursor.fetchall()
        return {"success": True, "rows": results}
    except sqlite3.Error as exc:
        return {"success": False, "error": str(exc)}
    finally:
        conn.close()


# Benchmark evaluation sample
schema_ddl = """
CREATE TABLE customers (id INT PRIMARY KEY, name TEXT);
CREATE TABLE orders (id INT PRIMARY KEY, customer_id INT, total REAL, status TEXT);
"""

seed_data = """
INSERT INTO customers VALUES (1, 'Acme Corp'), (2, 'Globex');
INSERT INTO orders VALUES (101, 1, 450.0, 'completed'), (102, 1, 120.0, 'pending'), (103, 2, 800.0, 'completed');
"""

generated_sql = """
SELECT c.name, SUM(o.total) AS total_spent
FROM customers c
JOIN orders o ON c.id = o.customer_id
WHERE o.status = 'completed'
GROUP BY c.id
ORDER BY total_spent DESC;
"""

valid = validate_sql(generated_sql)
execution = execute_in_sandbox(schema_ddl, seed_data, generated_sql)

print(f"Syntax Validation: {'PASS' if valid else 'FAIL'}")
print(f"Sandbox Execution: {'PASS' if execution['success'] else 'FAIL'}")
print(f"Query Results: {execution.get('rows')}")
print("[+] Evaluation benchmark completed successfully.")